# 04 Semantic Router

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `04-semantic-router.ipynb`

In [ ]:
# Start coding here

In [3]:
# !pip install sentence-transformers
# !pip install pandas
# !pip install scikit-learn

In [4]:
# =====================================================
# Notebook 04
# Semantic Router
# =====================================================

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
documents_df = pd.read_csv("artifacts/document_metadata.csv")

documents_df.head()

,document_id,filename,department,doc_type,created_at,content,char_count
0,1fe99400-06e1-4964-956b-3861687e7da4,AI_Roadmap.txt,engineering,corporate_document,2026-01-01,\n# AI Platform Roadmap\n\n## Search Infrastru...,516
1,b638264b-b788-442e-a09d-19176466fba8,Annual_Report_2026.txt,finance,corporate_document,2026-01-01,\n# Annual Financial Report 2026\n\n## Revenue...,479
2,31516a79-2cf5-4080-ac66-66b338d6ebff,Benefits_2026.txt,hr,corporate_document,2026-01-01,\n# Employee Benefits 2026\n\n## Health Insura...,610


In [6]:
documents_df["department"].value_counts()

department
engineering    1
finance        1
hr             1
Name: count, dtype: int64

In [7]:
department_profiles = {
    "hr": """
    Human Resources.
    Benefits.
    Healthcare.
    Leave Policy.
    Hiring.
    Employees.
    Compensation.
    Recruitment.
    """,
    "finance": """
    Revenue.
    Budget.
    Profit.
    Expenses.
    Financial Reports.
    Investments.
    Accounting.
    Forecasting.
    """,
    "engineering": """
    Software.
    Architecture.
    AI.
    Infrastructure.
    Cloud.
    Security.
    Platform.
    Development.
    """,
}

In [8]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [9]:
department_embeddings = {}

for department, text in department_profiles.items():

    embedding = embedding_model.encode(text)

    department_embeddings[department] = embedding

In [10]:
for department in department_embeddings:

    print(department, department_embeddings[department].shape)

hr (384,)
finance (384,)
engineering (384,)


In [11]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for department, dept_embedding in department_embeddings.items():

        score = cosine_similarity(
            query_embedding.reshape(1, -1), dept_embedding.reshape(1, -1)
        )[0][0]

        scores[department] = score

    best_department = max(scores, key=scores.get)

    return best_department, scores

In [12]:
query = "What healthcare benefits are available?"

department, scores = route_query(query)

department

'hr'

In [13]:
scores

{'hr': np.float32(0.44816947),
 'finance': np.float32(0.099409014),
 'engineering': np.float32(0.090729706)}

In [14]:
query = "How much revenue did we generate in 2026?"

department, scores = route_query(query)

print(department)

scores

finance


{'hr': np.float32(0.16781074),
 'finance': np.float32(0.37394804),
 'engineering': np.float32(0.058161102)}

In [15]:
query = "Describe our vector search architecture."

department, scores = route_query(query)

print(department)

scores

engineering


{'hr': np.float32(0.07858004),
 'finance': np.float32(0.14385195),
 'engineering': np.float32(0.37196565)}

In [16]:
queries = [
    "What benefits do employees receive?",
    "What was annual revenue?",
    "How does hybrid retrieval work?",
    "What is the hiring process?",
    "Explain cloud infrastructure.",
]

In [17]:
results = []

for query in queries:

    department, scores = route_query(query)

    results.append({"query": query, "routed_to": department})

routing_df = pd.DataFrame(results)

routing_df

,query,routed_to
0,What benefits do employees receive?,hr
1,What was annual revenue?,finance
2,How does hybrid retrieval work?,hr
3,What is the hiring process?,hr
4,Explain cloud infrastructure.,engineering


In [18]:
router_index = pd.DataFrame(
    {
        "department": list(department_profiles.keys()),
        "description": list(department_profiles.values()),
    }
)

router_index

,department,description
0,hr,\n Human Resources.\n Benefits.\n Hea...
1,finance,\n Revenue.\n Budget.\n Profit.\n ...
2,engineering,\n Software.\n Architecture.\n AI.\n ...


In [19]:
router_index.to_csv("artifacts/router_index.csv", index=False)

In [20]:
import pickle

with open("artifacts/router_embeddings.pkl", "wb") as file:

    pickle.dump(department_embeddings, file)